# 实验结果汇总
使用 Notebook 聚合 `cwPSU/results` 目录中的 CSV，计算相同 `bitlength`、`mu`、`hw` 的平均指标并生成汇总表。

In [3]:
# Section 1 – Setup Paths and Imports
from pathlib import Path
import pandas as pd

RESULTS_DIR = (Path.cwd() / "results").resolve()
OUTPUT_PATH = Path.cwd() / "results_summary.csv"
MB = 1024 ** 2

print(f"读取结果目录: {RESULTS_DIR}")
print(f"输出汇总文件: {OUTPUT_PATH}")

读取结果目录: /home/jrj/code/FeUPSU/experiments/results
输出汇总文件: /home/jrj/code/FeUPSU/experiments/results_summary.csv


In [5]:
# Section 2 – Load CSV Results
records = []
for csv_path in sorted(RESULTS_DIR.glob("*.csv")):
    try:
        raw = pd.read_csv(csv_path, header=None, names=["metric", "value"])
    except pd.errors.EmptyDataError:
        continue
    if raw.empty:
        continue
    record = raw.set_index("metric")["value"].to_dict()
    record["source_file"] = csv_path.name
    records.append(record)

metrics_df = pd.DataFrame(records)
if metrics_df.empty:
    raise ValueError(f"在 {RESULTS_DIR} 中未发现可用的结果 CSV")

numeric_cols = metrics_df.columns.drop("source_file")
for col in numeric_cols:
    metrics_df[col] = pd.to_numeric(metrics_df[col], errors="coerce")

metrics_df["source_file"] = metrics_df["source_file"].astype("string")
print(f"读取到 {len(metrics_df)} 份实验记录")
metrics_df.head()

读取到 111 份实验记录


,bitlength,comm_OTSSS,comm_gal_keys,comm_params,comm_pk,comm_relin_keys,comm_request,comm_response,effective_lambda,ell,...,log_poly_modulus_degree,mu,task,time_extract_response,time_offline_comm,time_online_comm,time_server,time_server_extract,time_setup_crypto,source_file
0,32,358131,35269619,83,245827,737559,26546861,489009,19,148,...,13,1,0,110310,19,266,118,7,36,effective_19_hw_3_1.csv
1,32,358093,35266648,83,245458,737599,26548170,488934,19,148,...,13,1,0,110173,20,283,135,4,35,effective_19_hw_3_2.csv
2,32,358105,35264222,83,245826,736887,11119953,488674,19,62,...,13,1,0,110792,9,163,100,7,46,effective_19_hw_4_1.csv
3,32,358261,35268382,83,245754,737074,11120646,489041,19,62,...,13,1,0,111294,9,163,99,7,39,effective_19_hw_4_2.csv
4,32,387599,47598990,85,251569,995710,7614715,496325,19,39,...,13,1,0,164012,6,188,133,10,61,effective_19_hw_5_1.csv


In [6]:
# Section 3 – Aggregate by Parameters
group_cols = ["effective_lambda", "hw"]
missing = [col for col in group_cols if col not in metrics_df.columns]
if missing:
    raise KeyError(f"缺少必要列: {missing}")

aggregated_df = (
    metrics_df
    .groupby(group_cols, dropna=False)
    .mean(numeric_only=True)
    .reset_index()
 )

print(f"聚合后得到 {len(aggregated_df)} 组参数")
aggregated_df.head()

聚合后得到 61 组参数


,effective_lambda,hw,bitlength,comm_OTSSS,comm_gal_keys,comm_params,comm_pk,comm_relin_keys,comm_request,comm_response,ell,log_poly_modulus_degree,mu,task,time_extract_response,time_offline_comm,time_online_comm,time_server,time_server_extract,time_setup_crypto
0,19,3,32.0,358112.0,35268133.5,83.0,245642.5,737579.0,26547515.5,488971.5,148.0,13.0,1.0,0.0,110241.5,19.5,274.5,126.5,5.5,35.5
1,19,4,32.0,358183.0,35266302.0,83.0,245790.0,736980.5,11120299.5,488857.5,62.0,13.0,1.0,0.0,111043.0,9.0,163.0,99.5,7.0,42.5
2,19,5,32.0,387363.5,47599150.5,85.0,250831.0,994207.5,7610632.0,497930.5,39.0,13.0,1.0,0.0,163405.0,6.5,204.0,139.5,7.0,61.0
3,19,6,32.0,386301.5,47600910.5,85.0,250889.5,992753.0,5851231.5,497595.0,30.0,13.0,1.0,0.0,164157.0,4.0,219.0,167.5,3.5,57.5
4,19,7,32.0,387021.5,47599677.0,85.0,250343.5,993101.5,5080118.5,498011.0,26.0,13.0,1.0,0.0,164475.5,4.5,242.0,174.0,7.0,47.0


In [7]:
# Section 4 – Compute Communication Metrics
summary_df = aggregated_df.copy()

focus_cols = ["time_server", "comm_request"]
for col in focus_cols:
    if col not in summary_df.columns:
        summary_df[col] = float("nan")

summary_df.head()

,effective_lambda,hw,bitlength,comm_OTSSS,comm_gal_keys,comm_params,comm_pk,comm_relin_keys,comm_request,comm_response,ell,log_poly_modulus_degree,mu,task,time_extract_response,time_offline_comm,time_online_comm,time_server,time_server_extract,time_setup_crypto
0,19,3,32.0,358112.0,35268133.5,83.0,245642.5,737579.0,26547515.5,488971.5,148.0,13.0,1.0,0.0,110241.5,19.5,274.5,126.5,5.5,35.5
1,19,4,32.0,358183.0,35266302.0,83.0,245790.0,736980.5,11120299.5,488857.5,62.0,13.0,1.0,0.0,111043.0,9.0,163.0,99.5,7.0,42.5
2,19,5,32.0,387363.5,47599150.5,85.0,250831.0,994207.5,7610632.0,497930.5,39.0,13.0,1.0,0.0,163405.0,6.5,204.0,139.5,7.0,61.0
3,19,6,32.0,386301.5,47600910.5,85.0,250889.5,992753.0,5851231.5,497595.0,30.0,13.0,1.0,0.0,164157.0,4.0,219.0,167.5,3.5,57.5
4,19,7,32.0,387021.5,47599677.0,85.0,250343.5,993101.5,5080118.5,498011.0,26.0,13.0,1.0,0.0,164475.5,4.5,242.0,174.0,7.0,47.0


In [8]:
# Section 5 – Display Aggregated Metrics
display_cols = group_cols + ["time_server", "comm_request"]

report_df = (
    summary_df[display_cols]
    .sort_values(group_cols)
    .reset_index(drop=True)
 )
report_df

,effective_lambda,hw,time_server,comm_request
0,19,3,126.5,26547515.5
1,19,4,99.5,11120299.5
2,19,5,139.5,7610632.0
3,19,6,167.5,5851231.5
4,19,7,174.0,5080118.5
...,...,...,...,...
56,50,15,866.0,34019357.0
57,50,16,919.0,32129845.0
58,50,17,1139.0,35978029.0
59,50,18,1192.0,34292079.0


In [ ]:
# Section 6 – Persist Processed Data
report_df.to_csv(OUTPUT_PATH, index=False)
print(f"已保存汇总结果至 {OUTPUT_PATH}")